# E45 — a barra que a amostra ergue sozinha

**A tentativa.** Ler a diferença de milésimos entre a entrega medida (E01) e a conta do
corte: 5,135% contra 5,138%. Para ler uma diferença dessas falta a barra --- e a fórmula da
raiz pede a lei na mão, que é justamente o que não se tem.

**O que se mede.** A barra erguida por reamostragem, com e sem datas:

1. no dado real, as frações de dez mil re-sorteios dos dias em que o corte existia, e a
   barra de 95% delas --- de dias e de blocos de 60 dias;
2. o exame de cobertura nos mundos que nunca mudam, onde a verdade é a conta do corte
   $k/(n+1)$: em que fração dos mundos cada barra contém a verdade.

**Convenções** (AGENTS.md §7 e §9): um experimento por caderno, parâmetros no topo marcados
com "brinque com", algoritmo em frevolab, resultado em lab/resultados/E45_reamostragem.json,
figura em .pdf e .png.

In [1]:
# <- brinque com: SERIE, JANELA, CAUDA, BLOCO, REAMOSTRAGENS, MUNDOS, REAMOSTRAGENS_MUNDOS, CONFIANCA, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, graficos, promessa, proporcao, volatilidade

SERIE = "sp500.csv"        # a série do arquivo do projeto anterior (.old/dados/)
JANELA = 252               # o corte: um ano de pregões
CAUDA = 0.05               # a promessa anunciada: 1 dia em 20
BLOCO = 60                 # o bloco que tem data
REAMOSTRAGENS = 10000      # re-sorteios do dado real
MUNDOS = 200               # mundos que nunca mudam (a mesma rotina do E01)
REAMOSTRAGENS_MUNDOS = 1000  # re-sorteios por mundo, no exame de cobertura
CONFIANCA = 0.95           # a confiança declarada das barras
SEMENTE = 112

retornos = volatilidade.retornos_log(dados.carregar_serie(SERIE))
violacoes = promessa.violacoes(retornos, JANELA, CAUDA).to_numpy().astype(float)
VERDADE = promessa.entrega_do_corte(JANELA, CAUDA)
print("frevolab %s | %s: %d dias | corte existiu em %d dias" % (
    frevolab.VERSAO, SERIE, len(retornos), violacoes.size))
print("entrega medida %.4f%% (%d violações) | a conta do corte %.4f%%" % (
    100 * violacoes.mean(), int(violacoes.sum()), 100 * VERDADE))

frevolab 0.1.0 | sp500.csv: 6718 dias | corte existiu em 6466 dias
entrega medida 5.1345% (332 violações) | a conta do corte 5.1383%


## A barra no dado real

In [2]:
# A barra erguida sozinha: as frações dos re-sorteios, de dias e de blocos inteiros.
rng = np.random.default_rng(SEMENTE)
fracao_dia = proporcao.reamostragens(violacoes, REAMOSTRAGENS, rng)
fracao_bloco = proporcao.reamostragens(violacoes, REAMOSTRAGENS, rng, bloco=BLOCO)

entrega = float(violacoes.mean())
barra_dia = proporcao.barra_reamostrada(violacoes, CONFIANCA, REAMOSTRAGENS, rng=rng)
barra_bloco = proporcao.barra_reamostrada(violacoes, CONFIANCA, REAMOSTRAGENS, rng=rng, bloco=BLOCO)

largura_dia_pp = 100 * (barra_dia[1] - barra_dia[0])
largura_bloco_pp = 100 * (barra_bloco[1] - barra_bloco[0])
largura_razao = largura_bloco_pp / largura_dia_pp
conta_dentro = bool(barra_dia[0] <= VERDADE <= barra_dia[1])

print("barra de dias:   [%.4f%%, %.4f%%]  largura %.4f pp" % (
    100 * barra_dia[0], 100 * barra_dia[1], largura_dia_pp))
print("barra de blocos: [%.4f%%, %.4f%%]  largura %.4f pp" % (
    100 * barra_bloco[0], 100 * barra_bloco[1], largura_bloco_pp))
print("razão de larguras: %.3f | a conta do corte cai %s a barra de dias" % (
    largura_razao, "dentro de" if conta_dentro else "fora de"))

barra de dias:   [4.6087%, 5.6758%]  largura 1.0671 pp
barra de blocos: [4.1121%, 6.3084%]  largura 2.1963 pp
razão de larguras: 2.058 | a conta do corte cai dentro de a barra de dias


In [3]:
# Figura 1: o histograma das reamostras de dias, com a barra e as duas linhas do capítulo.
fig, eixo = plt.subplots(figsize=(8.6, 4.1))
eixo.hist(100 * fracao_dia, bins=60, color="#1f4e79", alpha=0.85)
eixo.axvspan(100 * barra_dia[0], 100 * barra_dia[1], color="#c78f2c", alpha=0.25,
             label="a barra de 95%%: [%.2f%%, %.2f%%]" % (100 * barra_dia[0], 100 * barra_dia[1]))
eixo.axvline(100 * entrega, color="#1f4e79", lw=1.6,
             label="a entrega medida: %.3f%%" % (100 * entrega))
eixo.axvline(100 * VERDADE, color="#b03a2e", ls="--", lw=1.6,
             label="a conta do corte %d/%d: %.3f%% --- %s" % (
                 promessa.posto(JANELA, CAUDA), JANELA + 1, 100 * VERDADE,
                 "cai dentro da barra" if conta_dentro else "cai fora da barra"))
eixo.set_xlabel("taxa de violação das reamostras (%)")
eixo.set_ylabel("re-sorteios")
eixo.legend(frameon=False, fontsize=9)
graficos.salvar(fig, "E45_reamostragem", 1)
plt.close(fig)
print("figura 1 gravada")

figura 1 gravada


In [4]:
# Figura 2: as duas barras da mesma amostra, num eixo comum.
fig, eixo = plt.subplots(figsize=(8.6, 2.7))
eixo.plot([100 * barra_dia[0], 100 * barra_dia[1]], [0, 0], lw=10, color="#1f4e79",
          solid_capstyle="butt", label="dias: largura %.2f pp" % largura_dia_pp)
eixo.plot([100 * barra_bloco[0], 100 * barra_bloco[1]], [1, 1], lw=10, color="#c78f2c",
          solid_capstyle="butt", label="blocos de %d dias: largura %.2f pp" % (BLOCO, largura_bloco_pp))
eixo.axvline(100 * VERDADE, color="#b03a2e", ls="--", lw=1.4, label="a conta do corte")
eixo.set_yticks([0, 1])
eixo.set_yticklabels(["dias", "blocos de %d dias" % BLOCO])
eixo.set_ylim(-0.7, 1.7)
eixo.set_xlabel("taxa de violação (%)")
eixo.set_title("a barra de blocos é %.1f vezes mais larga que a de dias" % largura_razao,
               fontsize=10)
eixo.legend(frameon=False, fontsize=9, loc="lower right")
graficos.salvar(fig, "E45_reamostragem", 2)
plt.close(fig)
print("figura 2 gravada")

figura 2 gravada


## O exame nos mundos que nunca mudam

In [5]:
# A mesma rotina do E01: mundos de retornos independentes, sempre a mesma lei,
# submetidos ao mesmo corte. A verdade de cada um é a conta do corte k/(n+1).
rng_mundos = np.random.default_rng(SEMENTE + 1)
linhas = []
for _ in range(MUNDOS):
    mundo = pd.Series(rng_mundos.normal(0.0, 0.01, len(retornos)), index=retornos.index)
    linhas.append(promessa.violacoes(mundo, JANELA, CAUDA).to_numpy().astype(float))
matriz = np.vstack(linhas)
print("matriz de %d mundos por %d dias | violação média %.4f%% | a verdade %.4f%%" % (
    matriz.shape[0], matriz.shape[1], 100 * matriz.mean(), 100 * VERDADE))

matriz de 200 mundos por 6466 dias | violação média 5.1366% | a verdade 5.1383%


In [6]:
# O exame de cobertura: em quantos mundos a barra contém a verdade.
rng_exame = np.random.default_rng(SEMENTE + 2)
cobertura_dia = proporcao.cobertura(matriz, VERDADE, CONFIANCA, REAMOSTRAGENS_MUNDOS,
                                    rng=rng_exame, bloco=1)
cobertura_bloco = proporcao.cobertura(matriz, VERDADE, CONFIANCA, REAMOSTRAGENS_MUNDOS,
                                      rng=rng_exame, bloco=BLOCO)
mundos_dia = int(round(cobertura_dia * MUNDOS))
mundos_bloco = int(round(cobertura_bloco * MUNDOS))
margem = 100 * 1.96 * np.sqrt(CONFIANCA * (1 - CONFIANCA) / MUNDOS)  # o ruído de contagem
print("barra de dias:   contém a verdade em %.1f%% dos mundos (%d de %d)" % (
    100 * cobertura_dia, mundos_dia, MUNDOS))
print("barra de blocos: contém a verdade em %.1f%% dos mundos (%d de %d)" % (
    100 * cobertura_bloco, mundos_bloco, MUNDOS))
print("ruído de contagem: ±%.1f pontos percentuais" % margem)

# Figura 3: as duas coberturas contra a linha dos 95%.
fig, eixo = plt.subplots(figsize=(6.8, 3.8))
eixo.bar([0], [100 * cobertura_dia], 0.55, color="#1f4e79",
         label="dias: %d de %d mundos" % (mundos_dia, MUNDOS))
eixo.bar([1], [100 * cobertura_bloco], 0.55, color="#c78f2c",
         label="blocos de %d dias: %d de %d mundos" % (BLOCO, mundos_bloco, MUNDOS))
eixo.axhline(100 * CONFIANCA, color="#b03a2e", ls="--", lw=1.4,
             label="os %d%% prometidos" % round(100 * CONFIANCA))
eixo.axhspan(100 * CONFIANCA - margem, 100 * CONFIANCA + margem, color="#7f7f7f",
             alpha=0.2, label="o ruído de contagem")
eixo.set_xticks([0, 1])
eixo.set_xticklabels(["dias", "blocos de %d dias" % BLOCO])
eixo.set_ylabel("mundos em que a barra contém a verdade (%)")
eixo.set_ylim(80, 104)
eixo.legend(frameon=False, fontsize=8, loc="lower left")
graficos.salvar(fig, "E45_reamostragem", 3)
plt.close(fig)
print("figura 3 gravada")

barra de dias:   contém a verdade em 100.0% dos mundos (200 de 200)
barra de blocos: contém a verdade em 100.0% dos mundos (200 de 200)
ruído de contagem: ±3.0 pontos percentuais
figura 3 gravada


## Leitura visual das figuras

**Declarada contra os .png depois da execução** (AGENTS.md §9): este caderno foi entregue
construído e ainda não executado, e leitura visual sem figura é invenção. Quando rodar, a
leitura se declara aqui --- editar esta célula não invalida o resultado, porque o critério
de frescor é o hash das células de código.

O que as legendas do capítulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: histograma concentrado em torno de duas linhas praticamente coladas (a
   entrega medida e a conta do corte), as duas dentro da faixa da barra de 95%.
2. **Figura 2**: o traço de blocos visivelmente mais comprido que o de dias, os dois
   contendo a linha tracejada da conta do corte.
3. **Figura 3**: as duas barras no alto do eixo, acima ou rente à linha dos 95%, dentro ou
   na borda da faixa cinza do ruído de contagem.

In [7]:
# O resultado: um objeto por grandeza, em português, para o livro citar por comando.
resultado = {
    "boot_reamostragens": REAMOSTRAGENS,
    "boot_reamostragens_mundos": REAMOSTRAGENS_MUNDOS,
    "boot_cobertura_dia_pct": round(100 * cobertura_dia, 1),
    "boot_cobertura_bloco_pct": round(100 * cobertura_bloco, 1),
    "boot_cobertura_margem_pct": round(margem, 1),
    "boot_largura_dia_pp": round(largura_dia_pp, 2),
    "boot_largura_bloco_pp": round(largura_bloco_pp, 2),
    "boot_largura_razao": round(largura_razao, 1),
    "boot_mundos_cobertos_dia": mundos_dia,
    "boot_mundos_cobertos_bloco": mundos_bloco,
}
caminho = Path("lab/resultados/E45_reamostragem.json")
caminho.write_text(json.dumps(resultado, indent=1, ensure_ascii=False, sort_keys=True),
                   encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1, sort_keys=True))

{
 "boot_cobertura_bloco_pct": 100.0,
 "boot_cobertura_dia_pct": 100.0,
 "boot_cobertura_margem_pct": 3.0,
 "boot_largura_bloco_pp": 2.2,
 "boot_largura_dia_pp": 1.07,
 "boot_largura_razao": 2.1,
 "boot_mundos_cobertos_bloco": 200,
 "boot_mundos_cobertos_dia": 200,
 "boot_reamostragens": 10000,
 "boot_reamostragens_mundos": 1000
}
